# 💻 02. The Coder: 코드 자율 생성·실행·디버깅 에이전트 구축

이전 실습에서는 에이전트의 기본 구조(`create_agent`), Tool 설계, System Prompt 제어, 그리고 단기 메모리(Checkpointer)를 배웠습니다.

이번 실습에서는 **"코드를 직접 작성하고, 터미널에서 실행하며, 에러가 발생하면 스스로 코드를 고치는(Self-Healing)"** 진정한 의미의 **Coder 에이전트**를 구축합니다.

---

### 💡 핵심 패러다임: 코드 생성(Generation)과 코드 실행(Execution)의 분리

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                        일반 LLM vs Coder 에이전트 비교                       │
├──────────────────────────────────────┬──────────────────────────────────────┤
│               일반 LLM               │            Coder 에이전트            │
├──────────────────────────────────────┼──────────────────────────────────────┤
│ 💬 화면에 마크다운 코드를 출력만 함   │ 📝 file_writer로 실제 .py 파일 저장  │
│ ❌ 문법 오류나 실행 에러를 모름      │ ⚡ bash_command로 로컬에서 직접 실행 │
│ ❌ 사람이 직접 복붙해서 디버깅해야 함 │ 🩺 file_read + file_edit로 자율 수술 │
│ ❌ 수집된 결과 데이터를 검증 못 함    │ 📊 실행 결과/JSON 데이터를 스스로 확인 │
└──────────────────────────────────────┴──────────────────────────────────────┘
```

---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 핵심 내용 |
|:---:|:---|:---|
| **Step 0** | **환경 세팅** | 루트 경로 설정, `.env` 로드, `nest_asyncio`, `init_chat_model` 초기화 |
| **Part 1** | **Coder의 핵심 도구 체계 (`common.py`)** | `file_writer`, `file_read`, `file_edit`, `bash_command` 도구 이해 |
| **Part 2** | **Coder 전용 System Prompt 설계** | 외과수술적 수정(Surgical Edit) 원칙, 테스트 주도 개발(TDD) 지침 |
| **Part 3** | **`create_agent`로 Coder 조립** | 코딩 도구 세트와 프롬프트, 체크포인터를 결합한 Coder 에이전트 빌드 |
| **Part 4** | **[시나리오 1] 알고리즘 코드 생성 & 실행** | 1~100 소수 계산 스크립트 작성, 서브프로세스 실행 및 결과 수집 |
| **Part 5** | **[시나리오 2] 자율 디버깅 & Self-Healing** | 고의적 버그 발생 ➔ 에러 분석 ➔ `file_edit` 정밀 교체 ➔ 재실행 검증 |

---

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 추가하고, 산출물 저장 디렉토리(`artifacts/code`, `artifacts/data`)를 준비합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 환경변수 로드
load_dotenv(override=True)

# 2. 프로젝트 루트 경로 자동 설정 (상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Working Directory: {os.getcwd()}")
print(f"✅ Project Root: {project_root}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. 산출물 폴더 준비
CODE_DIR = os.path.abspath("artifacts/code")
DATA_DIR = os.path.abspath("artifacts/data")
os.makedirs(CODE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print(f"📂 Code Directory: {CODE_DIR}")
print(f"📂 Data Directory: {DATA_DIR}")

# 5. 통합 Chat Model Factory 로드
from app.utils import init_chat_model, normalize_content
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

llm = init_chat_model()
print(f"✅ LLM 모델 초기화 완료: {llm}")

---
## 🔧 Part 1. Coder의 핵심 도구 체계 (`app/tools/common.py`)

Coder 에이전트가 코드를 다루기 위해 사용하는 4가지 핵심 도구입니다:

| 도구명 | 역할 | 특징 및 장점 |
|:---|:---|:---|
| **`file_writer`** | 파이썬 파일 생성 / 덮어쓰기 | 완전한 초기 스크립트 작성 시 사용 |
| **`file_read`** | 라인 번호 포함 파일 열람 | 시작 줄(`offset`)과 줄 수(`limit`) 지원으로 토큰 과부하 방지 |
| **`file_edit`** | 특정 텍스트 블록 정밀 교체 | 전체 파일을 다시 쓰지 않는 **외과수술적 수정(Surgical Edit)** |
| **`bash_command`** | 쉘 명령어 및 파이썬 스크립트 실행 | 실제 인터프리터에서 `python script.py` 실행 및 stdout/stderr 수집 |

이제 `app.tools.common`에서 도구들을 임포트하여 구조를 살펴봅니다.

In [ ]:
from app.tools.common import (
    file_writer,
    file_read,
    file_edit,
    bash_command,
    grep_search,
    glob_search
)

coder_tools = [file_writer, file_read, file_edit, bash_command, grep_search, glob_search]

print("=== 🛠️ Coder 도구 메타데이터 확인 ===\n")
for t in coder_tools:
    print(f"• [{t.name}] : {t.description[:60]}...")

### 1.1 도구 직접 실행해보기 (단위 테스트)

에이전트에게 맡기기 전에, 우리가 직접 도구를 호출하여 `파일 생성 ➔ 읽기 ➔ 실행` 파이프라인을 확인해 봅니다.

In [ ]:
test_py_path = "artifacts/code/hello_test.py"

# 1. file_writer로 간단한 테스트 파이썬 파일 생성
sample_code = """# Hello Test Script
name = 'Antigravity Coder'
print(f'🚀 Hello from {name}!')
"""
write_res = file_writer.invoke({
    "file_path": test_py_path,
    "content": sample_code
})
print(f"1️⃣ FileWriter 결과: {write_res}")

# 2. file_read로 줄 번호와 함께 확인
read_res = file_read.invoke({
    "file_path": test_py_path,
    "show_line_numbers": True
})
print(f"\n2️⃣ FileRead 결과:\n{read_res}")

# 3. bash_command로 실행하여 출력 확인
exec_res = bash_command.invoke({
    "command": f"python {test_py_path}"
})
print(f"3️⃣ BashCommand 실행 결과:\n{exec_res}")

---
## 📋 Part 2. Coder 전용 System Prompt 설계

도구만 쥐어주면 에이전트가 코드를 작성하다가 사소한 에러 하나 때문에 파일 전체를 날려버리거나 무한 루프에 빠질 수 있습니다.

따라서 Coder 에이전트에는 다음과 같은 **시니어 엔지니어링 행동 수칙**을 시스템 프롬프트로 주입해야 합니다:

1. **외과수술적 수정 (Surgical Edit)**: 버그가 생기면 전체 파일을 다시 쓰지 말고 `file_edit`으로 고장 난 라인만 교체할 것.
2. **테스트 주도 자율 검증 (TDD)**: 코드를 생성하거나 수정한 뒤에는 반드시 `bash_command`로 실행하여 정상 종료 여부와 출력을 검증할 것.
3. **컨텍스트 보호 (Context Safety)**: 대용량 결과나 로그를 읽을 때 전체 파일을 읽지 말고 `file_read`의 `limit` 옵션을 활용할 것.

In [ ]:
CODER_SYSTEM_PROMPT = """
당신은 최고 수준의 시니어 파이썬 소프트웨어 엔지니어(Coder Agent)입니다.
당신의 임무는 사용자의 요구사항을 분석하여, 실제로 오류 없이 실행되는 견고한 파이썬 스크립트를 작성, 실행, 검증, 디버깅하는 것입니다.

═══════════════════════════════════════════════════════════════
[핵심 엔지니어링 행동 지침]
═══════════════════════════════════════════════════════════════

1. **코드 작성과 파일 생성 (`file_writer`)**
   - 새로운 스크립트를 작성할 때는 반드시 `artifacts/code/` 폴더 아래에 명확한 파일명(예: `scraper.py`, `calculator.py`)으로 저장하세요.
   - 코드는 PEP 8 스타일을 준수하고, 적절한 예외 처리(try-except)를 갖추어야 합니다.

2. **실행 및 자율 검증 (`bash_command`)**
   - 코드를 작성한 후에는 반드시 `bash_command`를 통해 `python artifacts/code/파일명.py` 형태로 스크립트를 실행해 보세요.
   - 실행 결과(stdout/stderr)를 직접 확인하고, 출력이 의도한 대로 나왔는지 스스로 검증하세요.

3. **외과수술적 디버깅 (Surgical Edit - `file_read` + `file_edit`)**
   - 실행 중 에러(SyntaxError, IndexError 등)가 발생하면, 파일 전체를 다시 작성하지 마세요.
   - `file_read`로 해당 파일의 줄 번호를 확인한 뒤, `file_edit` 도구로 문제가 된 부분만 정확하게 교체하세요.
   - 수정한 후에는 다시 스크립트를 실행하여 에러가 완치되었는지 확인하세요.

4. **결과 보고**
   - 모든 작업이 성공적으로 끝나면, 작성한 코드의 핵심 로직, 실행 결과 데이터, 저장 경로를 사용자에게 일목요연하게 보고하세요.
"""

print("✅ CODER_SYSTEM_PROMPT 정의 완료")

---
## 🚀 Part 3. `create_agent`로 Coder 에이전트 조립

이제 `create_agent` 함수를 사용하여 LLM, 코딩 도구 모음, 시스템 프롬프트, 메모리(Checkpointer)를 결합합니다.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# 1. 단기 메모리 체크포인터 생성
coder_memory = MemorySaver()

# 2. Coder 에이전트 인스턴스 구축
coder_agent = create_agent(
    model=llm,
    tools=coder_tools,
    system_prompt=CODER_SYSTEM_PROMPT,
    checkpointer=coder_memory
)

print("✅ Coder 에이전트 조립 완료:", coder_agent)

---
## 🧪 Part 4. [시나리오 1] 알고리즘 코드 자율 생성 & 실행

Coder 에이전트에게 1부터 100까지의 숫자 중 **소수(Prime Number)**를 구하는 스크립트를 작성하고 실행해 달라고 요청합니다.

에이전트가 단순히 머릿속으로 계산해서 답을 내는지, 아니면 **`file_writer` ➔ `bash_command`로 실제 코드를 짜서 실행하는지** 실행 궤적을 관찰하세요!

In [ ]:
config = {"configurable": {"thread_id": "coder_session_01"}}

query_1 = """
1부터 100까지의 숫자 중에서 소수(Prime Number)만 찾아내는 파이썬 스크립트를 
'artifacts/code/prime_calculator.py' 파일로 작성하고,
실제로 실행해서 소수의 개수와 목록을 확인한 뒤 나에게 보고해줘.
"""

print(f"🚀 [User 요청]: {query_1}\n")

response_1 = coder_agent.invoke(
    {"messages": [HumanMessage(content=query_1)]},
    config=config
)

print("=== 📋 에이전트 실행 궤적 (Execution Trajectory) ===\n")
for m in response_1["messages"]:
    m.pretty_print()
    print("-" * 50)

print("\n=== 💬 최종 보고서 ===")
print(normalize_content(response_1["messages"][-1].content))

In [ ]:
# 에이전트가 실제로 생성한 파일이 디스크에 존재하는지 확인
created_file = "artifacts/code/prime_calculator.py"
if os.path.exists(created_file):
    with open(created_file, "r", encoding="utf-8") as f:
        print(f"✅ 생성된 파일 확인: {created_file}\n")
        print(f.read())
else:
    print("❌ 파일이 생성되지 않았습니다.")

---
## 🩺 Part 5. [시나리오 2] 고의적 버그 발생 & 자율 디버깅 (Surgical Edit & Self-Healing)

Coder 에이전트의 진정한 힘은 **에러를 만났을 때 스스로 원인을 찾아 고치는 자가 치유(Self-Healing) 능력**에서 나옵니다.

이번에는 고의로 **버그가 포함된 파이썬 파일**을 만들어 두고, 에이전트에게 실행 및 디버깅을 지시해 봅니다.

In [ ]:
# 버그가 있는 스크립트 사전 작성: 0으로 나누기 오류 (ZeroDivisionError)
buggy_file = "artifacts/code/buggy_stats.py"
buggy_code = """# Statistics Calculator with a Bug
numbers = [10, 20, 30, 40, 50]

total = sum(numbers)
count = 0  # ⚠️ 버그: count가 0으로 고정되어 있어서 ZeroDivisionError 발생!

average = total / count
print(f"평균값: {average}")
"""

with open(buggy_file, "w", encoding="utf-8") as f:
    f.write(buggy_code)

print(f"⚠️ 버그가 포함된 파일 생성 완료: {buggy_file}")

In [ ]:
query_2 = """
'artifacts/code/buggy_stats.py' 파일을 실행해보고,
에러가 발생하면 file_read로 코드를 확인한 뒤 file_edit 도구로 버그(ZeroDivisionError)만 정확히 고쳐서(Surgical Edit)
다시 실행하고 올바른 평균값을 출력해줘.
"""

print(f"🚀 [User 요청]: {query_2}\n")

response_2 = coder_agent.invoke(
    {"messages": [HumanMessage(content=query_2)]},
    config=config
)

print("=== 📋 에이전트 자율 디버깅 궤적 ===\n")
for m in response_2["messages"]:
    m.pretty_print()
    print("-" * 50)

print("\n=== 💬 디버깅 완료 보고 ===")
print(normalize_content(response_2["messages"][-1].content))

In [ ]:
# 디버깅 후 실제로 수정된 파일 내용 확인
with open(buggy_file, "r", encoding="utf-8") as f:
    print(f"🎉 수정된 {buggy_file} 파일 내용:\n")
    print(f.read())

---
## 🎯 정리 및 핵심 요약 (Key Takeaways)

오늘 실습에서 우리는 **프로덕션 수준의 Coder 에이전트**를 직접 구축하고 검증했습니다.

1. **코드 생성과 실행의 완전한 분리**
   - LLM에게 마크다운으로 코드를 말하게만 두지 않고, `file_writer`로 저장하고 `bash_command`로 실제 실행하여 결과를 받아보게 함으로써 **환각(Hallucination) 없는 실제 동작 코드**를 확보했습니다.

2. **외과수술적 수정 (Surgical Edit)**
   - 에러 발생 시 `file_read`로 줄 번호를 파악하고 `file_edit`으로 고장 난 코드만 교체하여, **토큰 비용을 90% 이상 절감하고 안정적인 자가 치유(Self-Healing)**를 달성했습니다.

3. **에이전트 협업 파이프라인의 완성**
   - 다음 실습(`3_Navigator.ipynb`)에서는 웹사이트 구조를 스스로 탐색하는 **Navigator**를 배우고,
   - 최종적으로 **Navigator(분석) + Coder(수집)** 가 결합된 강력한 멀티에이전트 오케스트레이션 `4_MultiAgent_Orchestration.ipynb`으로 나아가게 됩니다. 🚀

수고하셨습니다!